# LeetCode 1736 — Latest Time by Replacing Hidden Digits bằng Python

Link đề: https://leetcode.com/problems/latest-time-by-replacing-hidden-digits/

Độ khó: **Easy** · Chủ đề: String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def maximumTime(self, time: str) -> str:

```

Mục tiêu học:

1. Đọc và xử lý chuỗi có định dạng cố định `hh:mm` bằng **chỉ số** (`time[0]`, `time[1]`, `time[3]`, `time[4]`).
2. Hiểu **chuỗi trong Python là bất biến** (immutable) → đổi sang `list`, sửa, rồi `"".join(...)`.
3. Giải bằng **vét cạn có thứ tự** (thử từ `23:59` lùi về `00:00`) — cách an toàn, dễ chứng minh.
4. Giải bằng **tham lam (greedy)**: chọn chữ số lớn nhất có thể cho từng vị trí, từ trái sang phải — và hiểu vì sao các vị trí **phụ thuộc nhau**.
5. Làm chủ hai bẫy: `?` ở hàng chục giờ phụ thuộc hàng đơn vị, và ngược lại.
6. Dùng định dạng `f"{h:02d}"` để in số có 2 chữ số.

## 1. Bài toán yêu cầu gì?

Cho chuỗi `time` dạng `hh:mm`, một số chữ số bị ẩn bằng dấu `?`.

Giờ hợp lệ nằm trong đoạn `00:00` … `23:59`. Hãy thay mọi `?` bằng chữ số sao cho kết quả là **giờ hợp lệ muộn nhất**.

| time | Kết quả | Vì sao? |
|---|---|---|
| `"2?:?0"` | `"23:50"` | giờ bắt đầu bằng 2 thì muộn nhất là 23; phút kết thúc bằng 0 muộn nhất là 50 |
| `"0?:3?"` | `"09:39"` | giờ `0?` muộn nhất là 09; phút `3?` muộn nhất là 39 |
| `"1?:22"` | `"19:22"` | giờ `1?` muộn nhất là 19 |

Constraints:

- `time` có định dạng `hh:mm`.
- Đề **đảm bảo** luôn tạo được ít nhất một giờ hợp lệ.

### Bố cục chỉ số của chuỗi

```text
chỉ số:  0   1   2   3   4
ký tự:   h1  h2  :   m1  m2
ví dụ:   2   ?   :   ?   0
```

- `time[0]`: hàng chục giờ (0, 1, 2).
- `time[1]`: hàng đơn vị giờ (0..9, nhưng nếu hàng chục là 2 thì chỉ 0..3).
- `time[2]`: luôn là `':'`.
- `time[3]`: hàng chục phút (0..5).
- `time[4]`: hàng đơn vị phút (0..9).

In [1]:
time = "2?:?0"
for i, ch in enumerate(time):
    print(f"time[{i}] = {ch!r}")

time[0] = '2'
time[1] = '?'
time[2] = ':'
time[3] = '?'
time[4] = '0'


## 2. Nền tảng Python cần biết

### 2.1. Truy cập ký tự và so sánh với `'?'`

`time[i]` trả một chuỗi dài 1. So sánh `time[i] == '?'` để biết vị trí đó có bị ẩn hay không.

In [2]:
time = "0?:3?"
print(time[0] == '?', time[1] == '?', time[3] == '?', time[4] == '?')

False True False True


### 2.2. Chuỗi là bất biến (immutable)

Không thể gán `time[1] = '9'`. Python báo `TypeError`. Ta bắt lỗi bằng `try/except` để thấy thông báo mà notebook vẫn chạy tiếp.

In [3]:
time = "1?:22"
try:
    time[1] = '9'
except TypeError as e:
    print("Lỗi:", e)

Lỗi: 'str' object does not support item assignment


### 2.3. Đổi sang `list` → sửa → `"".join`

`list("1?:22")` tách thành list ký tự. Sửa phần tử list thoải mái. `"".join(lst)` ghép lại thành chuỗi (chuỗi `""` là "keo" đặt giữa các phần tử — ở đây là rỗng).

In [4]:
t = list("1?:22")
print(t)
t[1] = '9'
print(t)
print("".join(t))
print("-".join(["a", "b", "c"]))   # keo là '-'

['1', '?', ':', '2', '2']
['1', '9', ':', '2', '2']
19:22
a-b-c


### 2.4. Toán tử `in` với chuỗi

`ch in "0123"` kiểm tra `ch` có là một trong các ký tự `'0','1','2','3'` không. Rất tiện cho điều kiện "hàng đơn vị giờ ≤ 3".

In [5]:
for ch in ['0', '3', '4', '9', '?']:
    print(ch, "in '0123?' ->", ch in "0123?")

0 in '0123?' -> True
3 in '0123?' -> True
4 in '0123?' -> False
9 in '0123?' -> False
? in '0123?' -> True


### 2.5. So sánh chữ số dạng ký tự

Các ký tự `'0'..'9'` có thứ tự giống số: `'3' < '9'` là `True`. Nhưng `'10' < '9'` cũng là `True` (so từng ký tự) — nên chỉ dùng khi so **một** chữ số.

In [6]:
print('3' < '9', '2' <= '3', '4' <= '3')
print('10' < '9', "<- cẩn thận: so chuỗi, không phải so số")

True True False
True <- cẩn thận: so chuỗi, không phải so số


### 2.6. Định dạng số 2 chữ số: `f"{h:02d}"`

`:02d` nghĩa là: số nguyên (`d`), độ rộng 2, thiếu thì **đệm số 0** bên trái. Dùng để tạo `"09"` từ `9`.

In [7]:
for h in [0, 9, 23]:
    print(f"{h:02d}")
print(f"{9:02d}:{5:02d}")

00
09
23
09:05


## 3. Cách trực giác — vét cạn từ muộn nhất về sớm nhất

Gợi ý của đề: "thử mọi đáp án từ lớn nhất đến nhỏ nhất". Chỉ có `24 × 60 = 1440` giờ, quá ít.

Ý tưởng:

1. Duyệt `h` từ 23 về 0, `m` từ 59 về 0.
2. Tạo chuỗi ứng viên `cand = f"{h:02d}:{m:02d}"`.
3. Kiểm tra `cand` **khớp** mẫu `time`: ở mỗi vị trí, `time[i] == '?'` hoặc `time[i] == cand[i]`.
4. Ứng viên đầu tiên khớp chính là đáp án (vì ta đi từ muộn về sớm).

### 3.1. Hàm kiểm tra khớp mẫu

`zip(a, b)` ghép từng cặp ký tự cùng vị trí. `all(...)` trả `True` nếu **mọi** điều kiện đúng.

In [8]:
print(list(zip("2?:?0", "23:50")))
print(all([True, True, False]), all([True, True]))

[('2', '2'), ('?', '3'), (':', ':'), ('?', '5'), ('0', '0')]
False True


In [9]:
def matches(pattern: str, cand: str) -> bool:
    return all(p == '?' or p == c for p, c in zip(pattern, cand))

print(matches("2?:?0", "23:50"))   # True
print(matches("2?:?0", "23:59"))   # False: vị trí 4 cần '0'
print(matches("2?:?0", "19:50"))   # False: vị trí 0 cần '2'

True
False
False


In [10]:
def maximum_time_brute(time: str) -> str:
    for h in range(23, -1, -1):          # 23, 22, ..., 0
        for m in range(59, -1, -1):      # 59, 58, ..., 0
            cand = f"{h:02d}:{m:02d}"
            if matches(time, cand):
                return cand
    return ""   # đề đảm bảo không xảy ra

print(maximum_time_brute("2?:?0"))
print(maximum_time_brute("0?:3?"))
print(maximum_time_brute("1?:22"))

23:50
09:39
19:22


### 3.2. `range(23, -1, -1)` đọc thế nào?

`range(start, stop, step)`: bắt đầu `23`, dừng **trước** `-1`, bước `-1`. Kết quả `23, 22, …, 0`. Viết `range(23, 0, -1)` sẽ **thiếu số 0**.

In [11]:
print(list(range(5, -1, -1)))
print(list(range(5, 0, -1)), "<- thiếu 0")

[5, 4, 3, 2, 1, 0]
[5, 4, 3, 2, 1] <- thiếu 0


### 3.3. Đếm số ứng viên phải thử

Vét cạn dừng ngay khi khớp, nhưng trường hợp xấu (đáp án sớm, ví dụ `"00:00"` không có `?`) phải thử gần đủ 1440 ứng viên.

In [12]:
def count_tries(time):
    tries = 0
    for h in range(23, -1, -1):
        for m in range(59, -1, -1):
            tries += 1
            if matches(time, f"{h:02d}:{m:02d}"):
                return tries
for t in ["??:??", "2?:?0", "00:00"]:
    print(t, "-> số ứng viên đã thử:", count_tries(t))

??:?? -> số ứng viên đã thử: 1
2?:?0 -> số ứng viên đã thử: 10
00:00 -> số ứng viên đã thử: 1440


## 4. Nút thắt và insight

**Nút thắt:** vét cạn đúng và đủ nhanh ở đây, nhưng nó không "hiểu" bài. Nếu định dạng dài hơn (ví dụ có giây, ngày, tháng), số ứng viên sẽ nổ tung.

**Insight (tham lam):** muốn giờ muộn nhất, hãy làm **chữ số bên trái lớn nhất có thể** trước — vì chữ số trái có "trọng số" lớn hơn mọi chữ số bên phải cộng lại. `2x:xx` luôn muộn hơn mọi `1x:xx`.

Nhưng "lớn nhất có thể" phải tính đến **ràng buộc hợp lệ**:

| Vị trí | Nếu là `?` thì chọn | Lý do |
|---|---|---|
| `time[0]` | `'2'` nếu `time[1]` là `?` hoặc ≤ `'3'`; ngược lại `'1'` | `24`..`29` không hợp lệ |
| `time[1]` | `'3'` nếu `time[0] == '2'`; ngược lại `'9'` | giờ tối đa `23` |
| `time[3]` | `'5'` | phút tối đa `59` |
| `time[4]` | `'9'` | luôn được |

Điểm tinh tế: **hai chữ số giờ phụ thuộc nhau**. Phút thì độc lập hoàn toàn.

In [13]:
# Thí nghiệm: vì sao không thể luôn đặt '2' cho time[0]?
for h2 in "0123456789":
    hour = int("2" + h2)
    print(f"2{h2} ->", "hợp lệ" if hour <= 23 else "KHÔNG hợp lệ")

20 -> hợp lệ
21 -> hợp lệ
22 -> hợp lệ
23 -> hợp lệ
24 -> KHÔNG hợp lệ
25 -> KHÔNG hợp lệ
26 -> KHÔNG hợp lệ
27 -> KHÔNG hợp lệ
28 -> KHÔNG hợp lệ
29 -> KHÔNG hợp lệ


## 5. Lời giải chính — tham lam từng vị trí

### 5.1. Bước 1: đổi chuỗi thành list

In [14]:
t = list("?4:5?")
print(t)

['?', '4', ':', '5', '?']


### 5.2. Bước 2: hàng chục giờ `t[0]`

Nếu `t[0] == '?'`: nhìn `t[1]`. Nếu `t[1]` là `?` (ta sẽ tự chọn được ≤ 3) hoặc là `'0'..'3'` → đặt `'2'`. Nếu `t[1]` là `'4'..'9'` → chỉ có thể đặt `'1'` (14..19).

In [15]:
t = list("?4:5?")
if t[0] == '?':
    t[0] = '2' if t[1] in "?0123" else '1'
print("sau bước 2:", "".join(t))

sau bước 2: 14:5?


### 5.3. Bước 3: hàng đơn vị giờ `t[1]`

**Sau khi** `t[0]` đã chắc chắn: nếu `t[0] == '2'` thì tối đa `'3'`, ngược lại (0 hoặc 1) tối đa `'9'`.

Thứ tự bước 2 → bước 3 quan trọng: với `"??"` bước 2 đặt `'2'`, bước 3 thấy `'2'` nên đặt `'3'` → `23`.

In [16]:
t = list("??:??")
if t[0] == '?':
    t[0] = '2' if t[1] in "?0123" else '1'
if t[1] == '?':
    t[1] = '3' if t[0] == '2' else '9'
print("giờ:", "".join(t[:2]))

giờ: 23


### 5.4. Bước 4: hai chữ số phút

Phút không phụ thuộc giờ: `t[3]` tối đa `'5'`, `t[4]` tối đa `'9'`.

### 5.5. Ghép lại

In [17]:
def maximum_time_greedy(time: str) -> str:
    t = list(time)
    if t[0] == '?':
        t[0] = '2' if t[1] in "?0123" else '1'
    if t[1] == '?':
        t[1] = '3' if t[0] == '2' else '9'
    if t[3] == '?':
        t[3] = '5'
    if t[4] == '?':
        t[4] = '9'
    return "".join(t)

for s in ["2?:?0", "0?:3?", "1?:22", "??:??", "?4:5?", "?3:??"]:
    print(s, "->", maximum_time_greedy(s))

2?:?0 -> 23:50
0?:3? -> 09:39
1?:22 -> 19:22
??:?? -> 23:59
?4:5? -> 14:59
?3:?? -> 23:59


## 6. TRACE từng bước

Bảng trace cho `"?4:?9"`:

| Vị trí | Ký tự trước | Điều kiện kiểm tra | Quyết định | Chuỗi sau |
|---:|:---:|---|---|---|
| 0 | `?` | `t[1]='4'` có trong `"?0123"`? Không | đặt `'1'` | `14:?9` |
| 1 | `4` | không phải `?` | giữ | `14:?9` |
| 3 | `?` | luôn | đặt `'5'` | `14:59` |
| 4 | `9` | không phải `?` | giữ | `14:59` |

In [18]:
def trace_maximum_time(time: str) -> str:
    t = list(time)
    print("ban đầu:", time)
    if t[0] == '?':
        cond = t[1] in "?0123"
        t[0] = '2' if cond else '1'
        print(f"  vị trí 0: t[1]={time[1]!r} thuộc '?0123'? {cond} -> đặt {t[0]!r} | {''.join(t)}")
    else:
        print(f"  vị trí 0: giữ {t[0]!r}")
    if t[1] == '?':
        cond = t[0] == '2'
        t[1] = '3' if cond else '9'
        print(f"  vị trí 1: t[0]=={t[0]!r} là '2'? {cond} -> đặt {t[1]!r} | {''.join(t)}")
    else:
        print(f"  vị trí 1: giữ {t[1]!r}")
    if t[3] == '?':
        t[3] = '5'
        print(f"  vị trí 3: đặt '5' | {''.join(t)}")
    else:
        print(f"  vị trí 3: giữ {t[3]!r}")
    if t[4] == '?':
        t[4] = '9'
        print(f"  vị trí 4: đặt '9' | {''.join(t)}")
    else:
        print(f"  vị trí 4: giữ {t[4]!r}")
    result = "".join(t)
    print("=> kết quả:", result)
    return result

trace_maximum_time("?4:?9")

ban đầu: ?4:?9
  vị trí 0: t[1]='4' thuộc '?0123'? False -> đặt '1' | 14:?9
  vị trí 1: giữ '4'
  vị trí 3: đặt '5' | 14:59
  vị trí 4: giữ '9'
=> kết quả: 14:59


'14:59'

### 6.1. Trace trường hợp "cả hai chữ số giờ đều ẩn"

Thấy rõ bước 1 quyết định `'2'` **trước**, rồi bước 2 mới dựa vào đó chọn `'3'`.

In [19]:
trace_maximum_time("??:??")
print()
trace_maximum_time("0?:3?")

ban đầu: ??:??
  vị trí 0: t[1]='?' thuộc '?0123'? True -> đặt '2' | 2?:??
  vị trí 1: t[0]=='2' là '2'? True -> đặt '3' | 23:??
  vị trí 3: đặt '5' | 23:5?
  vị trí 4: đặt '9' | 23:59
=> kết quả: 23:59

ban đầu: 0?:3?
  vị trí 0: giữ '0'
  vị trí 1: t[0]=='0' là '2'? False -> đặt '9' | 09:3?
  vị trí 3: giữ '3'
  vị trí 4: đặt '9' | 09:39
=> kết quả: 09:39


'09:39'

## 7. Code nộp LeetCode

In [20]:
class Solution:
    def maximumTime(self, time: str) -> str:
        t = list(time)
        if t[0] == '?':
            t[0] = '2' if t[1] in "?0123" else '1'
        if t[1] == '?':
            t[1] = '3' if t[0] == '2' else '9'
        if t[3] == '?':
            t[3] = '5'
        if t[4] == '?':
            t[4] = '9'
        return "".join(t)

sol = Solution()
print(sol.maximumTime("2?:?0"), sol.maximumTime("0?:3?"), sol.maximumTime("1?:22"))

23:50 09:39 19:22


## 8. Các lời giải khác

### 8.1. Vét cạn — bản nộp LeetCode

Đúng gợi ý của đề. 1440 ứng viên × 5 ký tự: rất nhanh.

In [21]:
class SolutionBrute:
    def maximumTime(self, time: str) -> str:
        for h in range(23, -1, -1):
            for m in range(59, -1, -1):
                cand = f"{h:02d}:{m:02d}"
                if all(p == '?' or p == c for p, c in zip(time, cand)):
                    return cand
        return ""

print(SolutionBrute().maximumTime("2?:?0"))

23:50


### 8.2. Tách giờ và phút, vét cạn **riêng** từng phần

Giờ và phút độc lập → chọn giờ lớn nhất khớp `time[:2]` (24 ứng viên), phút lớn nhất khớp `time[3:]` (60 ứng viên). Tổng chỉ 84 lần thử thay vì 1440.

`time[:2]` là slicing: lấy chỉ số 0, 1. `time[3:]` lấy từ chỉ số 3 tới hết.

In [22]:
print("2?:?0"[:2], "2?:?0"[3:])

def best_part(pattern: str, limit: int) -> str:
    for v in range(limit, -1, -1):
        cand = f"{v:02d}"
        if all(p == '?' or p == c for p, c in zip(pattern, cand)):
            return cand
    return ""

def maximum_time_split(time: str) -> str:
    return best_part(time[:2], 23) + ":" + best_part(time[3:], 59)

for s in ["2?:?0", "0?:3?", "1?:22", "??:??", "?4:5?"]:
    print(s, "->", maximum_time_split(s))

2? ?0
2?:?0 -> 23:50
0?:3? -> 09:39
1?:22 -> 19:22
??:?? -> 23:59
?4:5? -> 14:59


Code nộp LeetCode — tách giờ/phút:

In [23]:
class SolutionSplit:
    def maximumTime(self, time: str) -> str:
        def best(pattern: str, limit: int) -> str:
            for v in range(limit, -1, -1):
                cand = f"{v:02d}"
                if all(p == '?' or p == c for p, c in zip(pattern, cand)):
                    return cand
            return ""
        return best(time[:2], 23) + ":" + best(time[3:], 59)

print(SolutionSplit().maximumTime("?4:5?"))

14:59


### 8.3. Bảng so sánh

| Cách | Ý tưởng | Số phép thử (xấu nhất) | Time | Space | Khi nào dùng |
|---|---|---:|---:|---:|---|
| Vét cạn toàn bộ | thử 23:59 → 00:00 | 1440 | `O(1)` (hằng lớn) | `O(1)` | Chắc chắn đúng, dễ viết |
| Vét cạn tách phần | giờ, phút riêng | 84 | `O(1)` | `O(1)` | Khi các phần độc lập |
| Tham lam | chọn max từng vị trí | 4 | `O(1)` | `O(1)` | **Lời giải chính**, cần lý luận kỹ |

Vì độ dài chuỗi cố định là 5, mọi cách đều `O(1)`. Khác biệt nằm ở **hằng số** và **độ dễ chứng minh**.

## 9. Vì sao lời giải tham lam đúng?

**So sánh thời gian = so sánh chuỗi `hh:mm` theo từ điển**, vì mọi chuỗi cùng độ dài và cùng vị trí `':'`. Muốn chuỗi lớn nhất → làm ký tự trái nhất lớn nhất **mà vẫn còn hoàn thành được giờ hợp lệ**, rồi tới ký tự kế tiếp.

- **Vị trí 0:** `'2'` là lớn nhất có thể. Chọn `'2'` chỉ hoàn thành được nếu `time[1]` ∈ {`?`, `0..3`}. Nếu `time[1]` ∈ `4..9`, `'2'` bất khả → `'1'` (luôn hoàn thành được với mọi chữ số đơn vị). Không cần xét `'0'` vì `'1'` luôn khả thi và lớn hơn.
- **Vị trí 1:** sau khi đã cố định `t[0]`, giới hạn trên là `3` (nếu `t[0]='2'`) hoặc `9`. Chọn giới hạn đó.
- **Vị trí 3, 4:** phút không phụ thuộc giờ; `5` và `9` luôn hợp lệ.

Mỗi quyết định không làm mất khả năng hợp lệ của các vị trí sau, và là lớn nhất có thể → chuỗi kết quả lớn nhất.

Ta còn kiểm chứng bằng máy: so tham lam với vét cạn trên **mọi** mẫu có thể (mỗi vị trí là `?` hoặc một chữ số).

In [24]:
from itertools import product

checked = 0
for h1, h2, m1, m2 in product("?012", "?0123456789", "?012345", "?0123456789"):
    pattern = f"{h1}{h2}:{m1}{m2}"
    brute = maximum_time_brute(pattern)
    if brute == "":          # mẫu không tạo được giờ hợp lệ (vd "29:??") -> đề loại trừ
        continue
    assert maximum_time_greedy(pattern) == brute, pattern
    assert maximum_time_split(pattern) == brute, pattern
    checked += 1
print("Đã đối chiếu", checked, "mẫu hợp lệ: tham lam == vét cạn == tách phần")

Đã đối chiếu 2926 mẫu hợp lệ: tham lam == vét cạn == tách phần


`itertools.product(A, B, C, D)` sinh mọi bộ `(a, b, c, d)` với `a ∈ A`, `b ∈ B`… — tương đương 4 vòng `for` lồng nhau. Ví dụ nhỏ:

In [25]:
print(list(product("ab", "xy")))

[('a', 'x'), ('a', 'y'), ('b', 'x'), ('b', 'y')]


## 10. Độ phức tạp

Độ dài input cố định = 5.

- **Tham lam:** 4 phép kiểm tra → Time `O(1)`, Space `O(1)` (list 5 ký tự).
- **Vét cạn toàn bộ:** tối đa 1440 ứng viên × 5 so sánh → vẫn `O(1)` nhưng hằng số ~7200.
- **Tách phần:** tối đa 84 ứng viên → `O(1)`.

Nếu tổng quát hóa thành định dạng có `k` trường, vét cạn toàn bộ tăng theo **tích** các miền, còn tham lam chỉ tăng **tuyến tính** theo số vị trí.

## 11. Bộ test

Nhóm test:

1. Ba ví dụ của đề.
2. Không có `?` → trả nguyên chuỗi.
3. Toàn `?` → `"23:59"`.
4. Bẫy vị trí 0: `"?4:00"`, `"?9:00"` → `'1'`; `"?3:00"`, `"?0:00"` → `'2'`.
5. Bẫy vị trí 1: `"2?:00"` → `'3'`; `"1?:00"`, `"0?:00"` → `'9'`.
6. Phút: `"00:?0"`, `"00:5?"`, `"00:??"`.

In [26]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS ({actual})")

sol = Solution()
check("đề 1", sol.maximumTime("2?:?0"), "23:50")
check("đề 2", sol.maximumTime("0?:3?"), "09:39")
check("đề 3", sol.maximumTime("1?:22"), "19:22")
check("không có ?", sol.maximumTime("13:07"), "13:07")
check("toàn ?", sol.maximumTime("??:??"), "23:59")
check("?4 -> 14", sol.maximumTime("?4:00"), "14:00")
check("?9 -> 19", sol.maximumTime("?9:00"), "19:00")
check("?3 -> 23", sol.maximumTime("?3:00"), "23:00")
check("?0 -> 20", sol.maximumTime("?0:00"), "20:00")
check("2? -> 23", sol.maximumTime("2?:00"), "23:00")
check("1? -> 19", sol.maximumTime("1?:00"), "19:00")
check("0? -> 09", sol.maximumTime("0?:00"), "09:00")
check("phút ?0", sol.maximumTime("00:?0"), "00:50")
check("phút 5?", sol.maximumTime("00:5?"), "00:59")
check("phút ??", sol.maximumTime("00:??"), "00:59")

đề 1                   PASS (23:50)
đề 2                   PASS (09:39)
đề 3                   PASS (19:22)
không có ?             PASS (13:07)
toàn ?                 PASS (23:59)
?4 -> 14               PASS (14:00)
?9 -> 19               PASS (19:00)
?3 -> 23               PASS (23:00)
?0 -> 20               PASS (20:00)
2? -> 23               PASS (23:00)
1? -> 19               PASS (19:00)
0? -> 09               PASS (09:00)
phút ?0                PASS (00:50)
phút 5?                PASS (00:59)
phút ??                PASS (00:59)


In [27]:
# Đối chiếu 3 lớp Solution trên mọi mẫu hợp lệ
impls = [Solution(), SolutionBrute(), SolutionSplit()]
cnt = 0
for h1, h2, m1, m2 in product("?012", "?0123456789", "?012345", "?0123456789"):
    p = f"{h1}{h2}:{m1}{m2}"
    if maximum_time_brute(p) == "":
        continue
    outs = {s.maximumTime(p) for s in impls}
    assert len(outs) == 1, (p, outs)
    cnt += 1
print("3 bản nộp LeetCode khớp nhau trên", cnt, "mẫu")

3 bản nộp LeetCode khớp nhau trên 2926 mẫu


## 12. Lỗi người mới hay gặp

### Lỗi 1: Luôn đặt `'2'` cho `time[0]`

`"?5:00"` → `"25:00"` — không hợp lệ.

In [28]:
def wrong_always_two(time):
    t = list(time)
    if t[0] == '?': t[0] = '2'
    if t[1] == '?': t[1] = '3' if t[0] == '2' else '9'
    if t[3] == '?': t[3] = '5'
    if t[4] == '?': t[4] = '9'
    return "".join(t)

print("sai:", wrong_always_two("?5:00"), "| đúng:", maximum_time_greedy("?5:00"))

sai: 25:00 | đúng: 15:00


**Sửa:** chỉ đặt `'2'` khi `time[1]` là `?` hoặc ≤ `'3'`.

### Lỗi 2: Quên `'?'` trong điều kiện của vị trí 0

Viết `t[1] in "0123"` (thiếu `?`) → với `"??:??"` ra `"19:59"` thay vì `"23:59"`.

In [29]:
def wrong_missing_q(time):
    t = list(time)
    if t[0] == '?': t[0] = '2' if t[1] in "0123" else '1'
    if t[1] == '?': t[1] = '3' if t[0] == '2' else '9'
    if t[3] == '?': t[3] = '5'
    if t[4] == '?': t[4] = '9'
    return "".join(t)

print("sai:", wrong_missing_q("??:??"), "| đúng:", maximum_time_greedy("??:??"))

sai: 19:59 | đúng: 23:59


### Lỗi 3: Luôn đặt `'9'` cho `time[1]`

`"2?:00"` → `"29:00"` — không hợp lệ. **Sửa:** `'3'` khi hàng chục là `'2'`.

### Lỗi 4: Đặt `'9'` cho hàng chục phút

`"00:?0"` → `"00:90"` — phút tối đa 59. **Sửa:** hàng chục phút tối đa `'5'`.

In [30]:
def wrong_minute(time):
    t = list(time)
    if t[3] == '?': t[3] = '9'
    return "".join(t)

print("sai:", wrong_minute("00:?0"), "| đúng:", maximum_time_greedy("00:?0"))

sai: 00:90 | đúng: 00:50


### Lỗi 5: Cố gán trực tiếp vào chuỗi

`time[0] = '2'` → `TypeError: 'str' object does not support item assignment`. **Sửa:** `t = list(time)` rồi `"".join(t)`; hoặc ghép chuỗi mới bằng `+`.

### Lỗi 6: Dùng `range(23, 0, -1)` trong vét cạn

Bỏ sót giờ `00` → mẫu `"0?:??"` vẫn đúng (vì 09 có trong range), nhưng mẫu `"00:00"` sẽ trả `""`. Luôn dùng `stop = -1` khi muốn đi tới 0.

## 13. Mẹo nhận dạng pattern

- **"Lớn nhất / muộn nhất / nhỏ nhất theo thứ tự từ điển"** + chuỗi độ dài cố định → nghĩ **tham lam từ trái sang phải**: chữ số trái có trọng số lớn nhất.
- Trước khi chọn tham lam ở một vị trí, hỏi: "Lựa chọn này còn **hoàn thành** được không?" (ở đây: `2` ở hàng chục giờ chỉ ổn nếu hàng đơn vị ≤ 3).
- Khi không chắc tham lam đúng và **không gian nhỏ** (vài nghìn khả năng) → vét cạn có thứ tự là cách an toàn; còn dùng làm **oracle** để kiểm tra tham lam.
- Các trường **độc lập** (giờ và phút) → tách ra xử lý riêng.

Bài cùng họ: 949 Largest Time for Given Digits, 2437 Number of Valid Clock Times, 1323 Maximum 69 Number, 670 Maximum Swap.

## 14. Biến thể

### 14.1. Giờ **sớm nhất** hợp lệ

Đối xứng: mọi `?` đặt `'0'` là được (vì `00:00` hợp lệ và mọi chữ số đã biết đều ≥ 0). Nhưng cẩn thận: kiểm chứng bằng vét cạn tăng dần.

In [31]:
def earliest_time(time: str) -> str:
    return time.replace('?', '0')

def earliest_brute(time):
    for h in range(24):
        for m in range(60):
            c = f"{h:02d}:{m:02d}"
            if matches(time, c):
                return c
    return ""

for s in ["2?:?0", "?4:5?", "??:??"]:
    print(s, "->", earliest_time(s), "| brute:", earliest_brute(s))

2?:?0 -> 20:00 | brute: 20:00
?4:5? -> 04:50 | brute: 04:50
??:?? -> 00:00 | brute: 00:00


`str.replace(old, new)` trả chuỗi mới, thay mọi `old` bằng `new` (chuỗi gốc không đổi).

### 14.2. Đếm số giờ hợp lệ có thể tạo (bài 2437)

Vét cạn 1440 ứng viên, đếm số cái khớp.

In [32]:
def count_valid_times(time: str) -> int:
    return sum(1 for h in range(24) for m in range(60) if matches(time, f"{h:02d}:{m:02d}"))

print(count_valid_times("?5:00"))   # 05, 15 -> 2
print(count_valid_times("0?:0?"))   # 10*10 = 100
print(count_valid_times("??:??"))   # 1440

2
100
1440


### 14.3. Định dạng 12 giờ `hh:mm` với giờ từ `01` đến `12`

Giới hạn giờ đổi → chỉ cần đổi miền vét cạn. Đây là lợi thế của cách vét cạn: đổi luật mà không phải nghĩ lại các trường hợp.

In [33]:
def latest_12h(time: str) -> str:
    for h in range(12, 0, -1):
        for m in range(59, -1, -1):
            c = f"{h:02d}:{m:02d}"
            if matches(time, c):
                return c
    return ""

print(latest_12h("??:??"))   # 12:59
print(latest_12h("0?:??"))   # 09:59
print(latest_12h("?5:??"))   # 05:59

12:59
09:59
05:59


## 15. Bài tập tự luyện

Làm trước khi xem gợi ý/đáp án. Các hàm TODO không được gọi khi chạy notebook.

### Bài 1 — Tính tay

Tham lam cho kết quả gì với: `"?2:4?"`, `"?7:??"`, `"2?:??"`?

### Bài 2 — Dự đoán output

`"".join(list("ab?d"))` và `"12:3?".replace("?", "9")` in ra gì?

### Bài 3 — Điền TODO: hàm tham lam

In [34]:
def exercise_maximum_time(time: str) -> str:
    # TODO 1: t = list(time)
    # TODO 2: xử lý t[0] dựa trên t[1]
    # TODO 3: xử lý t[1] dựa trên t[0] (đã chốt)
    # TODO 4: t[3] -> '5', t[4] -> '9'
    pass

# assert exercise_maximum_time("??:??") == "23:59"
# assert exercise_maximum_time("?4:5?") == "14:59"

### Bài 4 — Tự viết vét cạn tách phần

Viết `exercise_best_hour(pattern2)` trả giờ lớn nhất (dạng `"hh"`) khớp mẫu 2 ký tự.

In [35]:
def exercise_best_hour(pattern2: str) -> str:
    # TODO: duyệt 23 -> 0, so khớp từng ký tự
    pass

# assert exercise_best_hour("?4") == "14"
# assert exercise_best_hour("2?") == "23"

### Bài 5 — Biến thể: giờ muộn nhất có **giây** `hh:mm:ss`

Mở rộng tham lam cho chuỗi 8 ký tự. Giây giống phút.

In [36]:
def exercise_maximum_time_seconds(time: str) -> str:
    # TODO: giống bài chính, thêm t[6] và t[7]
    pass

# assert exercise_maximum_time_seconds("??:??:??") == "23:59:59"
# assert exercise_maximum_time_seconds("?5:4?:?1") == "15:49:51"

### Bài 6 — Mini challenge: giờ muộn nhất **sớm hơn** một mốc cho trước

Cho mẫu `time` và mốc `limit` (`"hh:mm"`), trả giờ khớp mẫu, **≤ limit**, muộn nhất. Không có thì trả `""`.

In [37]:
def exercise_latest_before(time: str, limit: str) -> str:
    # TODO: vét cạn có thứ tự, bỏ qua ứng viên > limit
    pass

# assert exercise_latest_before("??:??", "12:34") == "12:34"
# assert exercise_latest_before("2?:??", "19:00") == ""
# assert exercise_latest_before("1?:?0", "13:45") == "13:40"

## 16. Gợi ý 3 mức

**Bài 1** — M1: xét `t[1]` trước khi chốt `t[0]`. M2: `2` ≤ 3, `7` > 3. M3: `"22:49"`, `"17:59"`, `"23:59"`.

**Bài 2** — M1: `join` ghép lại nguyên vẹn. M2: `replace` thay mọi `?`. M3: `"ab?d"`, `"12:39"`.

**Bài 3** — M1: thứ tự xử lý 0 → 1 → 3 → 4. M2: `t[1] in "?0123"`. M3: chép đúng bảng ở mục 4.

**Bài 4** — M1: `range(23, -1, -1)`. M2: `f"{h:02d}"`. M3: `all(p == '?' or p == c for p, c in zip(pattern2, cand))`.

**Bài 5** — M1: giây có cùng giới hạn với phút. M2: chỉ số giây là 6 và 7. M3: `t[6] → '5'`, `t[7] → '9'`.

**Bài 6** — M1: chuỗi `hh:mm` so sánh được trực tiếp bằng `<=`. M2: duyệt từ muộn về sớm, bỏ ứng viên `> limit`. M3: `if cand <= limit and matches(time, cand): return cand`.

## 17. Đáp án tham khảo

**Bài 1:** `"22:49"`, `"17:59"`, `"23:59"`. **Bài 2:** `"ab?d"`, `"12:39"`.

In [38]:
print(maximum_time_greedy("?2:4?"), maximum_time_greedy("?7:??"), maximum_time_greedy("2?:??"))
print("".join(list("ab?d")), "12:3?".replace("?", "9"))

22:49 17:59 23:59
ab?d 12:39


In [39]:
def answer_maximum_time(time):
    t = list(time)
    if t[0] == '?':
        t[0] = '2' if t[1] in "?0123" else '1'
    if t[1] == '?':
        t[1] = '3' if t[0] == '2' else '9'
    if t[3] == '?':
        t[3] = '5'
    if t[4] == '?':
        t[4] = '9'
    return "".join(t)


def answer_best_hour(pattern2):
    for h in range(23, -1, -1):
        cand = f"{h:02d}"
        if all(p == '?' or p == c for p, c in zip(pattern2, cand)):
            return cand
    return ""


def answer_maximum_time_seconds(time):
    t = list(time)
    if t[0] == '?':
        t[0] = '2' if t[1] in "?0123" else '1'
    if t[1] == '?':
        t[1] = '3' if t[0] == '2' else '9'
    for i in (3, 6):
        if t[i] == '?':
            t[i] = '5'
    for i in (4, 7):
        if t[i] == '?':
            t[i] = '9'
    return "".join(t)


def answer_latest_before(time, limit):
    for h in range(23, -1, -1):
        for m in range(59, -1, -1):
            cand = f"{h:02d}:{m:02d}"
            if cand <= limit and matches(time, cand):
                return cand
    return ""


assert answer_maximum_time("??:??") == "23:59"
assert answer_maximum_time("?4:5?") == "14:59"
assert answer_best_hour("?4") == "14"
assert answer_best_hour("2?") == "23"
assert answer_maximum_time_seconds("??:??:??") == "23:59:59"
assert answer_maximum_time_seconds("?5:4?:?1") == "15:49:51"
assert answer_latest_before("??:??", "12:34") == "12:34"
assert answer_latest_before("2?:??", "19:00") == ""
assert answer_latest_before("1?:?0", "13:45") == "13:40"
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maximumTime(self, time: str) -> str:
        t = list(time)                      # str bất biến -> list
        if t[0] == '?':
            t[0] = '2' if t[1] in "?0123" else '1'
        if t[1] == '?':
            t[1] = '3' if t[0] == '2' else '9'
        if t[3] == '?': t[3] = '5'
        if t[4] == '?': t[4] = '9'
        return "".join(t)
```

Template vét cạn có thứ tự (dùng khi không chắc tham lam):

```python
for cand in các_ứng_viên_từ_tốt_nhất_đến_tệ_nhất:
    if hợp_lệ(cand) and khớp_mẫu(cand):
        return cand
```

Checklist trước khi submit:

- [ ] Vị trí 0: đã xét cả `'?'` **và** `'0'..'3'` ở vị trí 1?
- [ ] Vị trí 1: xử lý **sau** khi đã chốt vị trí 0?
- [ ] Hàng chục phút tối đa `'5'`, không phải `'9'`?
- [ ] Không gán trực tiếp vào `str`; nhớ `"".join(t)`?
- [ ] Đã thử `"??:??"`, `"?4:00"`, `"2?:00"`?